# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
import os
from  dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
import tiktoken
from IPython.display import Markdown, display
from langsmith import Client

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
import langchain
print(langchain.__version__)

1.3.0


In [ ]:
load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search, 
        description="Web search tool"
        )
        ]

#agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Get the React prompt from hub
client = Client()
prompt = client.pull_prompt("hwchase17/react",
                            dangerously_pull_public_prompt=True
)
# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    llm=llm, 
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=(
    "Invalid ReAct format. If you have enough information, respond with "
    "Final Answer"
    "Otherwise use exactly 'Action:' and 'Action Input:'."
    ),
    max_execution_time=120,
    early_stopping_method="force"
)

In [53]:
from langchain_classic.agents.agent_types import AgentType

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [64]:
prompt_1 = """
Summarize the latest developments in generative AI models applied to healthcare in Markdown format with:
- Main heading: Latest Generative AI in healthcare (2026)
- Subheadings per key model or development
- Bullet points for features
- Source links if any
- The final answer should contain the full markdown summary
"""

response_1 = agent_executor.invoke({"input":prompt_1})



> Entering new AgentExecutor chain...
I need to gather the latest information on generative AI models applied to healthcare, focusing on developments in 2026. This will involve searching for recent articles or reports that summarize these advancements.

Action: TavilySafeSearch  
Action Input: "latest developments in generative AI models in healthcare 2026"  
[{'title': '5 Generative AI Milestones That Changed Healthcare This Year', 'url': 'https://www.youtube.com/watch?v=G5JdWttaKTs', 'content': '2026 may be remembered as the year healthcare AI started becoming practical. From ChatGPT Health and Claude Health to Microsoft Copilot', 'score': 0.9999502}, {'title': '2026 healthcare AI trends: Insights from experts | Wolters Kluwer', 'url': 'https://www.wolterskluwer.com/en/expert-insights/2026-healthcare-ai-trends-insights-from-experts', 'content': '### Italy\n\n### Netherlands\n\n### Norway\n\n### Poland\n\n### Portugal\n\n### Romania\n\n### Slovakia\n\n### Spain\n\n### Sweden\n\n### 

In [65]:
display(Markdown(response_1['output']))


```markdown
### Latest Generative AI in Healthcare (2026)

#### ChatGPT Health and Claude Health
- **Overview**: 2026 marks a significant year for practical applications of generative AI in healthcare.
- **Features**:
  - Enhanced patient communication.
  - Automation of clinical documentation.
  - Streamlined workflows for healthcare professionals.
- **Source**: [5 Generative AI Milestones That Changed Healthcare This Year](https://www.youtube.com/watch?v=G5JdWttaKTs)

#### Clinical-Grade Generative AI
- **Overview**: Generative AI is becoming a trusted partner in daily healthcare workflows.
- **Features**:
  - Automates documentation and synthesizes clinical notes.
  - Identifies care gaps and improves clinician-patient communication.
  - Focus on governance to manage "shadow AI" usage.
- **Source**: [2026 healthcare AI trends: Insights from experts | Wolters Kluwer](https://www.wolterskluwer.com/en/expert-insights/2026-healthcare-ai-trends-insights-from-experts)

#### Agentic Systems in Healthcare
- **Overview**: AI is evolving from standalone tools to intelligent systems that support clinicians.
- **Features**:
  - Improves clinical decision-making and workflow efficiency.
  - Allows clinicians to focus more on patient care.
- **Source**: [Emerging healthcare AI trends in 2026 so far | Philips](https://www.philips.com/a-w/about/news/archive/features/2026/this-years-emerging-trends-in-healthcare-ai-so-far.html)

#### Transformative Forces in Biomedical Research
- **Overview**: Generative AI is set to revolutionize biomedical research and therapeutic development.
- **Features**:
  - Uncovers hidden connections in scientific literature.
  - Accelerates the design of molecules and regulatory sequences.
  - Integrates multimodal data for proactive care coordination.
- **Source**: [Looking Ahead: Predictions for Artificial Intelligence and Medicine in 2026 | Mass General Brigham](https://www.massgeneralbrigham.org/en/about/newsroom/articles/2026-predictions-about-artificial-intelligence)

#### Reinventing Clinical Documentation
- **Overview**: Generative AI is enhancing the way clinical documentation is created.
- **Features**:
  - Drafts patient-friendly summaries and clinical notes.
  - Utilizes ambient listening and voice-to-text technologies.
- **Source**: [AI Trends For 2026 - How AI Is Rewiring Clinical Workflows and Medical Devices](https://mofotech.mofo.com/topics/ai-trends-for-2026---how-ai-is-rewiring-clinical-workflows-and-medical-devices)
```

### Latest Generative AI in Healthcare (2026)

#### ChatGPT Health and Claude Health
- **Overview**: 2026 marks a significant year for practical applications of generative AI in healthcare.
- **Features**:
  - Enhanced patient communication.
  - Automation of clinical documentation.
  - Streamlined workflows for healthcare professionals.
- **Source**: [5 Generative AI Milestones That Changed Healthcare This Year](https://www.youtube.com/watch?v=G5JdWttaKTs)

#### Clinical-Grade Generative AI
- **Overview**: Generative AI is becoming a trusted partner in daily healthcare workflows.
- **Features**:
  - Automates documentation and synthesizes clinical notes.
  - Identifies care gaps and improves clinician-patient communication.
  - Focus on governance to manage "shadow AI" usage.
- **Source**: [2026 healthcare AI trends: Insights from experts | Wolters Kluwer](https://www.wolterskluwer.com/en/expert-insights/2026-healthcare-ai-trends-insights-from-experts)

#### Agentic Systems in Healthcare
- **Overview**: AI is evolving from standalone tools to intelligent systems that support clinicians.
- **Features**:
  - Improves clinical decision-making and workflow efficiency.
  - Allows clinicians to focus more on patient care.
- **Source**: [Emerging healthcare AI trends in 2026 so far | Philips](https://www.philips.com/a-w/about/news/archive/features/2026/this-years-emerging-trends-in-healthcare-ai-so-far.html)

#### Transformative Forces in Biomedical Research
- **Overview**: Generative AI is set to revolutionize biomedical research and therapeutic development.
- **Features**:
  - Uncovers hidden connections in scientific literature.
  - Accelerates the design of molecules and regulatory sequences.
  - Integrates multimodal data for proactive care coordination.
- **Source**: [Looking Ahead: Predictions for Artificial Intelligence and Medicine in 2026 | Mass General Brigham](https://www.massgeneralbrigham.org/en/about/newsroom/articles/2026-predictions-about-artificial-intelligence)

#### Reinventing Clinical Documentation
- **Overview**: Generative AI is enhancing the way clinical documentation is created.
- **Features**:
  - Drafts patient-friendly summaries and clinical notes.
  - Utilizes ambient listening and voice-to-text technologies.
- **Source**: [AI Trends For 2026 - How AI Is Rewiring Clinical Workflows and Medical Devices](https://mofotech.mofo.com/topics/ai-trends-for-2026---how-ai-is-rewiring-clinical-workflows-and-medical-devices)

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [86]:
# Your prompt here
prompt_2 = """
You must track the top emerging AI startups of 2025 and their innovations.(URL and content)
- Main heading: Top emerging AI startups and innovations (2026)
- Subheadings per key model or development
- Bullet points summarizing the main innovations
- Source links if any
- The final answer should contain the full markdown summary
"""

# Run it
response_2 = agent_executor.invoke({"input":prompt_2})




> Entering new AgentExecutor chain...
To track the top emerging AI startups of 2025 and their innovations, I will perform a web search to gather the most recent information on this topic. 

Action: TavilySafeSearch  
Action Input: "top emerging AI startups 2025 innovations"  [{'title': 'Top 10 AI Startups in 2025 – The New Wave of AI Innovation', 'url': 'https://www.aitoolsfordailyuse.com/top-10-ai-startups-2025/', 'content': 'Learn about AI in customer support.\n\n## 10. Hugging Face – Open Source AI Startup 2025\n\nWhile now a major platform, Hugging Face started as a startup and continues to be the epicenter of open-source AI innovation. Its platform for sharing models and datasets is indispensable for researchers and developers, fueling countless other AI trends.  \n Innovation: The central repository for open-source AI.  \n Category: AI Infrastructure & Platform\n\nExplore Hugging Face on GitHub.\n\n## Conclusion\n\nThe most successful AI startups of 2025 aren’t just building ge

In [88]:
display(Markdown(response_2["output"]))


Agent stopped due to iteration limit or time limit.

# Top emerging AI startups and innovations (2026)

## 1. Hugging Face
- **Innovation**: Central repository for open-source AI.
- **Category**: AI Infrastructure & Platform.
- **Source**: [AI Tools for Daily Use](https://www.aitoolsfordailyuse.com/top-10-ai-startups-2025/)

## 2. Synthesia
- **Innovation**: No-shoot AI avatar video creation.
- **Category**: Corporate Training & Comms.
- **Source**: [AI Tools for Daily Use](https://www.aitoolsfordailyuse.com/top-10-ai-startups-2025/)

## 3. Cresta
- **Innovation**: Real-time AI coaching for customer service.
- **Category**: Enterprise SaaS.
- **Source**: [AI Tools for Daily Use](https://www.aitoolsfordailyuse.com/top-10-ai-startups-2025/)

## 4. Groq
- **Innovation**: Advanced hardware for efficient generative AI model operation.
- **Valuation**: $6.9B.
- **Source**: [Eqvista](https://eqvista.com/top-ai-startups-by-valuation/)

## 5. Mistral AI
- **Innovation**: Strong alternatives in AI model development.
- **Valuation**: $13.25B.
- **Source**: [Eqvista](https://eqvista.com/top-ai-startups-by-valuation/)

## 6. Cognition
- **Innovation**: AI coding agent technology.
- **Valuation**: $10B.
- **Source**: [Forbes](https://www.forbes.com/lists/ai50/)

## 7. Generative AI Tools
- **Innovation**: Tools for creating art, music, and stories with minimal input.
- **Source**: [YouTube](https://www.youtube.com/watch?v=Ax0LhF4s_CA)

These startups are leading the way in AI innovation, focusing on specific industries and applications to deliver significant value.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [ ]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}
'''
prompt = PromptTemplate.from_template(template)

agent = create_react_agent(llm, tools, prompt)

agent_executor_2 = AgentExecutor(
    tools=tools,
    llm=llm,
    agent=agent,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
    max_execution_time=180,
    early_stopping_method="generate",
    return_intermediate_steps=True,
)

In [ ]:
# Your prompt here
prompt_3 = """
Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Table header: Compare AI Models
- Table columns: models
- Table rows: comparison metrics
- Summarize the main characteristics of each metric
- Source links if any
- The final answer should contain the full summary in a table format
Final answer: must include ONLY the table comparing the different models
"""

# Run it
response_3 = agent_executor_2.invoke({"input":prompt_3})



> Entering new AgentExecutor chain...
I need to gather information about OpenAI's ChatGPT Team and Anthropic's Claude Pro, focusing on their key features, pricing, and reviews. This will involve searching for the latest details on both AI models.

Action: TavilySafeSearch  
Action Input: "OpenAI ChatGPT Team features pricing reviews"  
[{'title': 'ChatGPT Plans Compared: Free vs Plus ($20) vs Pro ($200) vs ...', 'url': 'https://intuitionlabs.ai/articles/chatgpt-plans-comparison', 'content': 'In January 2024, OpenAI filled the “middle” market with ChatGPT Teams (later renamed Business), a self-serve plan for small and mid-size teams (( "Highlights: Product")) (( "Highlights: How does the pricing work,for ChatGPT Business")). Teams offered collaboration (shared workspace, custom GPTs, admin tools) along with advanced models (GPT-4 with 32K context, premium DALL·E-3 images, browsing, data analysis) (( "Highlights: ChatGPT Team includes:")) (( "Highlights: ,to new features and improvemen

In [ ]:
display(Markdown(response_3["output"]))

| Feature/Metric                | OpenAI ChatGPT Team                          | Anthropic Claude Pro                          |
|-------------------------------|----------------------------------------------|-----------------------------------------------|
| **Pricing**                   | $25/user/month (annual) or $30 (monthly)    | $20/month (annual) or $20 (monthly)          |
| **User Minimum**              | 2 users                                      | 1 user                                       |
| **Workspace Type**            | Shared team workspace                        | Individual and team collaboration             |
| **Admin Controls**            | Yes                                          | Yes (in Team plans)                          |
| **Data Privacy**              | Never trained on user data by default       | Not specified, but generally strong privacy   |
| **Model Access**              | GPT-4 with advanced features                 | Claude with extended reasoning models         |
| **Custom GPTs**               | Create and share within workspace            | Available in Team plans                       |
| **Usage Limits**              | Higher usage caps than Plus                  | Increased usage compared to free tier        |
| **Key Features**              | Collaboration tools, admin tools, browsing, data analysis | File creation, code execution, Google Workspace integration |
| **Target Audience**           | Small to medium enterprises, teams           | Individual power users, small teams          |
| **Reviews**                   | Generally positive, noted for collaboration features | Positive, praised for formal writing and coding capabilities |
| **Source Links**              | [ChatGPT Plans Comparison](https://intuitionlabs.ai/articles/chatgpt-plans-comparison), [OpenAI Debuts ChatGPT Team](https://www.marketingaiinstitute.com/blog/chatgpt-team) | [Anthropic Claude Review](https://searchatlas.com/blog/anthropic-claude-review/), [Claude Pricing](https://www.finout.io/blog/claude-pricing-in-2026-for-individuals-organizations-and-developers) |

This table summarizes the key features, pricing, and reviews of both AI models, providing a clear comparison for potential users.

| Feature/Metric                | OpenAI ChatGPT Team                          | Anthropic Claude Pro                          |
|-------------------------------|----------------------------------------------|-----------------------------------------------|
| **Pricing**                   | $25/user/month (annual) or $30 (monthly)    | $20/month (annual) or $20 (monthly)          |
| **User Minimum**              | 2 users                                      | 1 user                                       |
| **Workspace Type**            | Shared team workspace                        | Individual and team collaboration             |
| **Admin Controls**            | Yes                                          | Yes (in Team plans)                          |
| **Data Privacy**              | Never trained on user data by default       | Not specified, but generally strong privacy   |
| **Model Access**              | GPT-4 with advanced features                 | Claude with extended reasoning models         |
| **Custom GPTs**               | Create and share within workspace            | Available in Team plans                       |
| **Usage Limits**              | Higher usage caps than Plus                  | Increased usage compared to free tier        |
| **Key Features**              | Collaboration tools, admin tools, browsing, data analysis | File creation, code execution, Google Workspace integration |
| **Target Audience**           | Small to medium enterprises, teams           | Individual power users, small teams          |
| **Reviews**                   | Generally positive, noted for collaboration features | Positive, praised for formal writing and coding capabilities |
| **Source Links**              | [ChatGPT Plans Comparison](https://intuitionlabs.ai/articles/chatgpt-plans-comparison), [OpenAI Debuts ChatGPT Team](https://www.marketingaiinstitute.com/blog/chatgpt-team) | [Anthropic Claude Review](https://searchatlas.com/blog/anthropic-claude-review/), [Claude Pricing](https://www.finout.io/blog/claude-pricing-in-2026-for-individuals-organizations-and-developers) |

This table summarizes the key features, pricing, and reviews of both AI models, providing a clear comparison for potential users.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
